<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-03-prompt-engineering/lesson-3.2-few-shot-and-in-context/notebooks/exercises-3.2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 3.2: Few-Shot & In-Context Learning — Practice Exercises

**Netsetos GenAI Course v2.0** | Module 3 | 8 exercises: format-vs-labels, number sweet spot, ordering bias, smart retrieval, many-shot, contrastive examples, format leakage, dynamic classifier.

---

In [ ]:
!pip install google-generativeai sentence-transformers numpy -q
import google.generativeai as genai
# genai.configure(api_key='YOUR_KEY')  # Get free key from aistudio.google.com
model = genai.GenerativeModel('gemini-2.0-flash')
def ask(prompt, temp=0.0):
    r = model.generate_content(prompt, generation_config={'temperature': temp})
    return r.text

---
## Exercise 1 (Easy): Format vs Labels — The Min et al. Test
Few-shot with CORRECT labels vs RANDOMLY SHUFFLED labels. Does it actually matter?

In [ ]:
test_set = [
    'Loved every minute, brilliant film',
    'Total waste of time, walked out',
    'Solid acting, slow plot',
    'Visually stunning but emotionally hollow',
    'Best Bollywood release this year',
]

# Correct few-shot
correct = '''Review: "Amazing performance, must-watch" -> positive
Review: "Boring and predictable" -> negative
Review: "It was alright" -> neutral

Review: "{r}" ->'''

# Randomly shuffled labels
shuffled = '''Review: "Amazing performance, must-watch" -> negative
Review: "Boring and predictable" -> neutral
Review: "It was alright" -> positive

Review: "{r}" ->'''

for r in test_set:
    a = ask(correct.format(r=r)).strip().split('\n')[0]
    b = ask(shuffled.format(r=r)).strip().split('\n')[0]
    print(f'{r[:35]:38} | correct: {a[:12]:12} | shuffled: {b[:12]}')
print('\nFormat IS the signal — accuracy stays high even with wrong labels.')

---
## Exercise 2 (Easy): How Many Examples? — Find the Sweet Spot
Classify the same 10 reviews with 0, 1, 3, 5, 10 examples. Plot the accuracy curve.

In [ ]:
examples_pool = [
    ('Excellent product, highly recommend', 'positive'),
    ('Terrible quality, broke immediately', 'negative'),
    ('It works fine for the price', 'neutral'),
    ('Best purchase of the year', 'positive'),
    ('Complete waste of money', 'negative'),
    ('Average, nothing special', 'neutral'),
    ('Loved it from day one', 'positive'),
    ('Disappointing, not as described', 'negative'),
    ('Decent enough, gets the job done', 'neutral'),
    ('Outstanding quality and service', 'positive'),
]

test = [
    ('Fantastic, exceeded my expectations', 'positive'),
    ('Awful, would not buy again', 'negative'),
    ('Just okay, no surprises', 'neutral'),
    ('Five stars, beautiful design', 'positive'),
    ('Broke after one use', 'negative'),
    ('Pretty standard for the category', 'neutral'),
    ('Best in class, no doubt', 'positive'),
    ('Subpar build, returning it', 'negative'),
    ('Nothing wrong but nothing great', 'neutral'),
    ('Top tier, worth every rupee', 'positive'),
]

for n in [0, 1, 3, 5, 10]:
    shots = '\n'.join(f'Review: "{t}" -> {l}' for t, l in examples_pool[:n])
    correct = 0
    for t, label in test:
        prompt = (shots + '\n' if shots else '') + f'Review: "{t}" ->'
        ans = ask(prompt).strip().lower().split('\n')[0]
        if label in ans:
            correct += 1
    print(f'{n}-shot: {correct}/10')
print('\nWatch for the plateau — 3-5 is usually the sweet spot.')

---
## Exercise 3 (Medium): Ordering Bias — Same Examples, Different Order
Shuffle the same 5 examples 5 times. Measure variance in output accuracy.

In [ ]:
import random
examples = [
    ('Loved the service', 'positive'),
    ('Worst experience ever', 'negative'),
    ('It was okay', 'neutral'),
    ('Excellent quality', 'positive'),
    ('Not impressed at all', 'negative'),
]
test = [('Quite enjoyed it', 'positive'), ('Pretty bad really', 'negative'),
        ('Average at best', 'neutral'), ('Absolutely brilliant', 'positive'),
        ('Disliked everything', 'negative')]

results = []
for run in range(5):
    random.seed(run)
    order = examples.copy()
    random.shuffle(order)
    shots = '\n'.join(f'Review: "{t}" -> {l}' for t, l in order)
    correct = 0
    for t, label in test:
        ans = ask(shots + f'\nReview: "{t}" ->').strip().lower().split('\n')[0]
        if label in ans:
            correct += 1
    results.append(correct)
    print(f'Run {run+1} (order={[l[0] for _, l in order]}): {correct}/5')

print(f'\nMin: {min(results)}, Max: {max(results)}, Range: {max(results)-min(results)}')
print('Variance > 1 means ordering matters — pin a good order in production.')

---
## Exercise 4 (Medium): Smart Example Selection via Embeddings
Index 12 examples; retrieve the 3 most similar to each query.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer('all-MiniLM-L6-v2')

pool = [
    ('Biryani was bland and overpriced', 'negative'),
    ('Loved the dosa, crispy and hot', 'positive'),
    ('Service was slow but food was great', 'neutral'),
    ('App keeps crashing, terrible UX', 'negative'),
    ('UPI transfer worked instantly', 'positive'),
    ('Order arrived on time, packaging fine', 'neutral'),
    ('Driver was rude and cab was dirty', 'negative'),
    ('Smooth ride, polite driver', 'positive'),
    ('OTP did not arrive, had to retry 3 times', 'negative'),
    ('Onboarding was easy, KYC done in 5 min', 'positive'),
    ('Discount applied as expected', 'neutral'),
    ('Customer support never picked up', 'negative'),
]

embeddings = encoder.encode([t for t, _ in pool])

def retrieve(query, k=3):
    qv = encoder.encode([query])[0]
    sims = embeddings @ qv / (np.linalg.norm(embeddings, axis=1) * np.linalg.norm(qv))
    idx = np.argsort(-sims)[:k]
    return [pool[i] for i in idx]

queries = ['food was tasteless', 'app refused to open', 'cab arrived early']
for q in queries:
    shots = retrieve(q, 3)
    prompt = '\n'.join(f'Review: "{t}" -> {l}' for t, l in shots) + f'\nReview: "{q}" ->'
    ans = ask(prompt).strip().split('\n')[0]
    print(f'Query: "{q}"')
    print(f'  Retrieved: {[t[:30] for t, _ in shots]}')
    print(f'  Answer: {ans}\n')

---
## Exercise 5 (Medium): Many-Shot — 30 Examples in One Prompt
Does more help? Compare 3-shot vs 30-shot on a harder task: detecting code-mixed Hindi-English sentiment.

In [ ]:
code_mix = [
    ('bahut achha tha', 'positive'), ('faltu hai', 'negative'), ('thik thaak', 'neutral'),
    ('mast product', 'positive'), ('paisa barbad', 'negative'), ('chalta hai', 'neutral'),
    ('zabardast service', 'positive'), ('ekdum bekaar', 'negative'), ('average bhai', 'neutral'),
    ('mind blowing', 'positive'), ('time waste', 'negative'), ('so so', 'neutral'),
    ('kya baat hai amazing', 'positive'), ('totally bekar', 'negative'), ('decent yaar', 'neutral'),
    ('ekdum top class', 'positive'), ('worst experience yaar', 'negative'), ('not bad', 'neutral'),
    ('superb quality', 'positive'), ('return karunga', 'negative'), ('okay-ish', 'neutral'),
    ('jhakaas product', 'positive'), ('paise wapas chahiye', 'negative'), ('fine hai', 'neutral'),
    ('dil khush ho gaya', 'positive'), ('mat lo bhai', 'negative'), ('chalega', 'neutral'),
    ('bahut badhiya', 'positive'), ('disappointed kar diya', 'negative'), ('theek hi hai', 'neutral'),
]

test = [('product mast hai bilkul', 'positive'), ('total fraud', 'negative'),
        ('average product hai', 'neutral'), ('paisa vasool', 'positive'),
        ('useless cheez', 'negative')]

for n in [3, 30]:
    shots = '\n'.join(f'"{t}" -> {l}' for t, l in code_mix[:n])
    correct = 0
    for t, label in test:
        ans = ask(shots + f'\n"{t}" ->').strip().lower().split('\n')[0]
        if label in ans:
            correct += 1
    print(f'{n}-shot: {correct}/5')
print('\nFor low-resource / code-mixed inputs, more examples often help more.')

---
## Exercise 6 (Hard): Contrastive Examples — Show What NOT to Do
Mix positive examples with explicit negative examples ("don't do this").

In [ ]:
# Task: extract a person's name in `Last, First` format only
contrastive_prompt = '''Extract the person's name in `Last, First` format.

GOOD:
Text: "Dr. Priya Sharma performed the surgery." -> Sharma, Priya
Text: "Reporter Arjun Reddy filed the story." -> Reddy, Arjun

BAD (do NOT do this):
Text: "Mr. Rohan Mehta arrived." -> Rohan Mehta   (wrong: full name, not Last, First)
Text: "Anita said hello." -> Anita   (wrong: only first name)

Now extract from this text:
Text: "{t}" ->'''

tests = [
    'Producer Karan Johar announced the film.',
    'CEO Sundar Pichai was interviewed.',
    'Defendant Vikram Singh pleaded guilty.',
    'Witness Meera Iyer testified for hours.',
]
for t in tests:
    print(f'{t}\n  -> {ask(contrastive_prompt.format(t=t)).strip().split(chr(10))[0]}\n')

print('Contrastive examples teach the model what to AVOID — useful for strict formats.')

---
## Exercise 7 (Hard): Format Leakage — When Examples Sabotage You
Show how a small format inconsistency in examples poisons every output.

In [ ]:
# Examples have INCONSISTENT format — some use {} JSON, some use plain text
leaky = '''Extract the city.
Text: "I am from Mumbai" -> {"city": "Mumbai"}
Text: "She lives in Pune" -> Pune
Text: "They moved to Bangalore" -> {"city": "Bangalore"}
Text: "He works in Hyderabad" -> Hyderabad

Text: "{t}" ->'''

# Examples with CONSISTENT JSON format
clean = '''Extract the city. Output JSON like {"city": "X"}.
Text: "I am from Mumbai" -> {"city": "Mumbai"}
Text: "She lives in Pune" -> {"city": "Pune"}
Text: "They moved to Bangalore" -> {"city": "Bangalore"}

Text: "{t}" ->'''

tests = ['I love Chennai', 'Born in Kolkata', 'Currently in Delhi']
print('LEAKY (mixed formats):')
for t in tests:
    print(f'  {t} -> {ask(leaky.format(t=t)).strip().split(chr(10))[0]}')
print('\nCLEAN (consistent JSON):')
for t in tests:
    print(f'  {t} -> {ask(clean.format(t=t)).strip().split(chr(10))[0]}')

print('\nInconsistent example format = unparseable production output.')

---
## Exercise 8 (Challenge): Dynamic Few-Shot Classifier with Caching
Combine retrieval-based example selection with a cacheable system prompt — production pattern.

In [ ]:
# Reuse the encoder + pool from Exercise 4
system_prefix = '''You are a sentiment classifier for Indian e-commerce reviews.
Output one word only: positive, negative, or neutral.
Code-mixed Hindi-English is normal. Match the most likely intent.
Below are dynamically-retrieved examples; the user's review is at the end.'''

def classify(review, k=3):
    shots = retrieve(review, k)
    examples_str = '\n'.join(f'Review: "{t}" -> {l}' for t, l in shots)
    prompt = f'{system_prefix}\n\n{examples_str}\n\nReview: "{review}" ->'
    ans = ask(prompt).strip().split('\n')[0].lower()
    for label in ['positive', 'negative', 'neutral']:
        if label in ans:
            return label
    return 'unknown'

production_inputs = [
    'app crash ho gayi 5 baar',
    'food was hot and tasty',
    'cab on time, no issues',
    'order missing items, returning',
    'package arrived as described',
]

print('Dynamic few-shot classifier:')
for inp in production_inputs:
    print(f'  "{inp[:35]:35}" -> {classify(inp)}')

print('\nIn production: cache `system_prefix` (Anthropic prompt caching).')
print('Per-call cost drops ~90% after first warm call within 5 min TTL.')

---
**8 exercises complete.** Few-shot mastery: format-vs-labels, sweet-spot tuning, ordering bias, smart retrieval, many-shot for low-resource inputs, contrastive examples, format leakage, production-grade dynamic classifier with caching.

**Next: Lesson 3.3 — Advanced Reasoning & Chain-of-Thought.**